# v3 수정 : HAVAA 결측 제거 (2026-10-11)

- 문제 : HAVAA 계수를 2025 학습기간 500구 이상 구종에만 적합 → 너클볼(KN)·스크루볼(SC) HAVAA 가 NaN → league.json 에 NaN 이 써져 사이트 데이터 로드 실패
- 수정 : `stuff_model.fit_group_coef` — 500구 이상 구종은 기존 계수 그대로(동일 확인), 적은 구종은 공통 기울기 + 자체 절편(30구 이상) 또는 공통 계수. 학습에 없던 구종도 공통 계수(default)로 계산
- LGBM·스케일러는 그대로 (재학습하면 LGBM 표본 추출 난수 때문에 KN·SC 외 투구 점수도 평균 1.2점 흔들려서 채택 안 함)
- 결과 : 트래킹이 있는 모든 투구의 HAVAA 결측 0. KN·SC 외 투구 점수는 완전히 같음
- Jupyter 커널 연결 문제로 같은 코드를 스크립트로 실행했다

In [ ]:
"""v3 수정 : 기존 LGBM 은 그대로 두고 HAVAA(·보정 HAA) 대체 계수만 추가해 결측 제거 (stuff_pipeline_261011_v3fix.ipynb 와 같은 코드)"""
import os, json, time, joblib
import numpy as np
import pandas as pd
from scipy import stats
import stuff_pipeline as sp
import stuff_model as sm
import export_web_data as ew

T0 = time.time()
log = lambda *a: print(f'[{time.time() - T0:5.0f}s]', *a, flush=True)
CACHE = os.path.join('outputs', 'vaa_exp', 'cache')
F_FEATS = [f for f in sp.STUFF_FEATURES if f not in ('vx0', 'vz0', 'ax', 'ay', 'az', 'vy0')]
cols = sorted(set(F_FEATS) | {'game_date', 'vaa', 'haa', 'plate_z', 'plate_x', 'waste', 'pitcher', 'description', 'type',
                              'estimated_woba_using_speedangle'})
fr = {y: pd.read_parquet(os.path.join(CACHE, f'frame_{y}.parquet'), columns=cols + (['y3', 'split'] if y == 2025 else []))
      for y in (2024, 2025, 2026)}
d = fr[2025]
valid_start = pd.Timestamp(pd.read_json(os.path.join(CACHE, 'meta.json'), typ='series')['valid_start'])
is_tr = (d['game_date'] < valid_start).to_numpy()
num = lambda s: s.to_numpy('float64', na_value=np.nan)

def add_raw(x):
    lefty = (x['p_throws'].astype(str) == 'L').to_numpy()
    x['haa_raw'] = np.where(lefty, -num(x['haa']), num(x['haa']))
    x['rel_x_raw'] = np.where(lefty, -num(x['release_pos_x']), num(x['release_pos_x']))
    return x

for y in fr:
    add_raw(fr[y])
fit = d[is_tr].copy()
fit['pitch_type'] = fit['pitch_type'].astype(str); fit['p_throws'] = fit['p_throws'].astype(str)
vaa_coef, vaa_def = sm.fit_group_coef(fit, 'vaa', ['plate_z'], ['pitch_type'])
haa_coef, haa_def = sm.fit_group_coef(fit, 'haa_raw', ['plate_x', 'rel_x_raw'], ['pitch_type', 'p_throws'])

old = {v: sm.load_model(os.path.join('models', v)) for v in ('v3', 'v3_h4')}
for k, b in old['v3'].vaa_coef.items():      # 500구 이상 구종은 기존 계수와 같아야 한다
    assert np.allclose(vaa_coef[k], b, rtol=0, atol=1e-10), k
for k, b in old['v3_h4'].haa_coef.items():
    assert np.allclose(haa_coef[k], b, rtol=0, atol=1e-10), k
n_tr = fit.groupby('pitch_type').size()
log('HAVAA 계수 :', {k[0]: ('자체' if k in old['v3'].vaa_coef else f'공통 기울기 + 자체 절편 (학습 {n_tr.get(k[0], 0)}구)') for k in vaa_coef
                    if k not in old['v3'].vaa_coef}, '/ 공통 계수', np.round(vaa_def, 4))
log('보정 HAA 추가 그룹 :', [k for k in haa_coef if k not in old['v3_h4'].haa_coef], '/ 공통 계수', np.round(haa_def, 4))
log('2025 학습기간 구종 수', fit.groupby('pitch_type').size().to_dict())

for y in fr:
    fr[y]['vaa_aa'] = sm._group_resid(fr[y], num(fr[y]['vaa']), [num(fr[y]['plate_z'])], ['pitch_type'], vaa_coef, vaa_def)
    fr[y]['haa_aa'] = sm._group_resid(fr[y], fr[y]['haa_raw'].to_numpy(), [num(fr[y]['plate_x']), fr[y]['rel_x_raw'].to_numpy()],
                                      ['pitch_type', 'p_throws'], haa_coef, haa_def)
    nt = sp.is_no_tracking(fr[y]).to_numpy()
    log(f'{y} HAVAA 결측 : 트래킹 있는 투구 {int(np.isnan(fr[y]["vaa_aa"][~nt]).sum())} / 트래킹 없는 투구 {int(nt.sum())}'
        f' · 보정 HAA 결측 (트래킹 있음) {int(np.isnan(fr[y]["haa_aa"][~nt]).sum())}')
    assert not np.isnan(fr[y]['vaa_aa'].to_numpy()[~nt]).any()

d = fr[2025]
samp = d[d['y3'].notna()]
tr_rows, es_rows = samp[samp['split'] == 'tr'], samp[samp['split'] == 'es']
specs = {'v3': (F_FEATS + ['vaa_aa'], None), 'v3_h4': (F_FEATS + ['vaa_aa', 'haa_aa'], haa_coef)}

def validity(x, s):
    x = x.assign(s=s, swing=x['description'].isin(ew.SWING_DESC), whiff=x['description'].isin(sp.WHIFF_DESC))
    pp = x.groupby(['pitcher', 'pitch_type'], observed=True).agg(n=('swing', 'size'), s=('s', 'mean'), sw=('swing', 'sum'), wh=('whiff', 'sum')).query('n >= 100')
    return stats.spearmanr(pp['s'], pp['wh'] / pp['sw']).statistic

for ver, (feats, hc) in specs.items():
    lgbm = old[ver].lgbm                       # 재학습하지 않는다 (KN·SC 외 투구 점수 불변)
    m = sm.AngleAdjustedStuffModel(lgbm, feats, vaa_coef, hc, vaa_default=vaa_def, haa_default=haa_def if hc else None)
    X = d[m.input_features]
    ref = d[F_FEATS].copy()
    for k, v in m.adjusted(d).items():
        ref[k] = v
    p = m.predict(X)
    assert np.array_equal(p, lgbm.predict(ref[feats])), ver
    for exact in (False, True):
        s = X.sample(20000, random_state=42)
        c, b = m.explain(s, exact)
        assert np.abs(b + c.sum(1) - m.predict(s)).max() < 1e-9, (ver, exact)
    scored_tr = is_tr & ~d['waste'].fillna(False).to_numpy().astype(bool) & ~sp.is_no_tracking(d).to_numpy()
    scaler = joblib.load(os.path.join('models', ver, 'stage4_stuff_scaler.joblib'))   # 스케일러도 그대로
    sc = scaler.transform(p)
    old_sc = joblib.load(os.path.join('models', ver, 'stage4_stuff_scaler.joblib'))
    # 기존 v3 와 비교
    for y in (2025, 2026):
        x = fr[y]
        new_s = scaler.transform(m.predict(x[m.input_features]))
        old_s = old_sc.transform(old[ver].predict(x[old[ver].input_features]))
        rare = x['pitch_type'].astype(str).isin(['KN', 'SC']).to_numpy()
        nt = sp.is_no_tracking(x).to_numpy()
        ok = ~rare & ~nt
        msg = (f'  {ver} {y} : 점수 상관 (KN·SC 제외) {np.corrcoef(new_s[ok], old_s[ok])[0, 1]:.5f}, 평균 |차이| {np.abs(new_s[ok] - old_s[ok]).mean():.3f}점'
               f', 최대 |차이| {np.abs(new_s[ok] - old_s[ok]).max():.2e}점, KN·SC {int(rare.sum())}구 평균 점수 {old_s[rare].mean():.1f} → {new_s[rare].mean():.1f}')
        if y == 2025:
            va = ~is_tr & ~d['waste'].fillna(False).to_numpy().astype(bool)
            msg += f' · 검증 ρ(구위, whiff%) {validity(x[va], old_s[va]):.4f} → {validity(x[va], new_s[va]):.4f}'
        log(msg)
    log(f'  {ver} 2025 학습기간 점수 중앙값 {np.median(sc[scored_tr]):.2f}')
    out = os.path.join('models', ver)
    joblib.dump(m, os.path.join(out, sm.MODEL_FILE))
    joblib.dump(scaler, os.path.join(out, 'stage4_stuff_scaler.joblib'), compress=3)
    json.dump({'features': feats, 'input_features': m.input_features}, open(os.path.join(out, 'features.json'), 'w'), indent=1)
    log(f'  저장 {out}')
log('끝')
